# Notebook 2: Fraud Intelligence Engine
### ET AI Hackathon 2026 - Digital Public Safety Platform (PS6)

FIXED VERSION - see "FIX-2026-07-10" comments in the code cells below for what changed and why.

This notebook implements the six-module Fraud Intelligence Engine:

1. Signal Extraction
2. Knowledge Retrieval (RAG)
3. LLM Reasoning (OpenRouter / tencent/hy3:free)
3b. Indicator Normalisation
4. Fraud Classification (Deterministic)
5. Evidence Validation
6. Rule-based Risk Engine

followed by full pipeline orchestration, sample test cases, notes/limitations, and a standalone deterministic test suite.

## 1. Imports and Logging Configuration

In [13]:
"""
Core imports for the Fraud Intelligence Engine.

chromadb              - reads the persistent vector store created in Notebook 1
sentence_transformers - encodes the user query with the same embedding model
                        used in Notebook 1, so vectors are comparable
requests              - OpenRouter API client used for the reasoning module
difflib               - lightweight fuzzy matching for indicator normalisation
"""

import difflib
import json
import logging
import os
import re
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Dict, List, Optional, Tuple

import chromadb
import requests
from sentence_transformers import SentenceTransformer

In [14]:
# ---------------------------------------------------------------------------
# Logging configuration
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("fraud_intelligence_engine")

## 2. Configuration

In [15]:
class Config:
    """
    Central configuration for Notebook 2.

    These values must stay consistent with Notebook 1's knowledge base
    build configuration.
    """

    # --- Knowledge base (must match Notebook 1) ---
    CHROMA_PERSIST_DIR: str = "./knowledge_vector_db"
    CHROMA_COLLECTION_NAME: str = "fraud_scam_knowledge_base"
    EMBEDDING_MODEL_NAME: str = "all-MiniLM-L6-v2"

    # --- Retrieval ---
    TOP_K: int = 5
    MIN_RELEVANCE_SCORE: float = 0.1

    # --- OpenRouter / tencent/hy3:free ---
    OPENROUTER_MODEL_NAME: str = "tencent/hy3:free"

    # FIX-2026-07-10: this must be the NAME of an environment variable,
    # never the key itself. The previous version had a real key pasted
    # here directly, which leaked it into version control / shared files.
    OPENROUTER_API_KEY_ENV_VAR: str = "OPENROUTER_API_KEY"

    OPENROUTER_API_URL: str = "https://openrouter.ai/api/v1/chat/completions"
    LLM_TEMPERATURE: float = 0.2

    # FIX-2026-07-10: root cause of the JSON truncation crash.
    # tencent/hy3:free is a reasoning model - its hidden "thinking" tokens
    # are drawn from the same max_tokens budget as the final answer unless
    # a reasoning-specific budget is set. With max_tokens=2048 the model
    # was spending almost everything on reasoning and getting cut off
    # mid-string while writing the "explanation" field.
    LLM_MAX_OUTPUT_TOKENS: int = 4096
    LLM_REASONING_MAX_TOKENS: int = 1024

    # If a response is still truncated (finish_reason == "length"), retry
    # once with this larger budget before giving up on a clean parse.
    LLM_RETRY_MAX_OUTPUT_TOKENS: int = 6144
    LLM_MAX_RETRIES: int = 1

    # --- Indicator normalisation ---
    INDICATOR_FUZZY_MATCH_CUTOFF: float = 0.6

    # --- Fraud classification ---
    CLASSIFICATION_MIN_OVERLAP_RATIO: float = 0.4
    UNCLASSIFIED_LABEL: str = "Unclassified Suspicious Activity"

    # --- Risk engine ---
    RISK_LOW_MAX: int = 30
    RISK_MEDIUM_MAX: int = 60
    RISK_HIGH_MAX: int = 85


CONFIG = Config()
logger.info(
    "Configuration loaded. persist_dir=%s collection=%s",
    CONFIG.CHROMA_PERSIST_DIR,
    CONFIG.CHROMA_COLLECTION_NAME,
)

2026-07-11 06:16:27,448 | INFO | fraud_intelligence_engine | Configuration loaded. persist_dir=./knowledge_vector_db collection=fraud_scam_knowledge_base


## 3. Module 1 - Signal Extraction

In [16]:
@dataclass
class SignalExtractionResult:
    """
    Structured output of Module 1.

    entities   - regex-extracted structured data points
    behaviours - boolean-style behavioural flags detected in the text
    raw_matches - the literal substrings that triggered each behavioural flag
    """

    entities: Dict[str, List[str]] = field(default_factory=dict)
    behaviours: Dict[str, bool] = field(default_factory=dict)
    raw_matches: Dict[str, List[str]] = field(default_factory=dict)

In [17]:
# --- Entity regex patterns -------------------------------------------------

_PATTERNS: Dict[str, str] = {
    "phone_numbers": r"\b(?:\+91[\-\s]?)?[6-9]\d{9}\b",
    "upi_ids": r"\b[\w.\-]{2,256}@[a-zA-Z]{2,64}\b",
    "pan_numbers": r"\b[A-Z]{5}[0-9]{4}[A-Z]\b",
    "aadhaar_numbers": r"\b\d{4}\s?\d{4}\s?\d{4}\b",
    "bank_accounts": r"\b\d{9,18}\b",
    "urls": r"https?://[^\s]+|www\.[^\s]+",
    "otp_mentions": r"\bOTP\b|\bone[- ]time password\b",
    "money_amounts": r"(?:₹|Rs\.?|INR)\s?[\d,]+(?:\.\d+)?(?:\s?(?:lakh|crore|k))?",
}

In [18]:
# --- Behavioural keyword groups ---------------------------------------------

_BEHAVIOUR_KEYWORD_GROUPS: Dict[str, List[str]] = {
    "authority_impersonation": [
        "cbi",
        "ed",
        "enforcement directorate",
        "rbi",
        "income tax department",
        "trai",
        "cyber cell",
        "cyber crime branch",
        "police department",
        "customs department",
        "narcotics",
        "supreme court",
        "high court",
    ],
    "threat_language": [
        "arrest warrant",
        "you will be arrested",
        "legal action",
        "fir will be filed",
        "case will be registered",
        "jail",
        "criminal case",
        "non bailable",
    ],
    "urgency": [
        "immediately",
        "right now",
        "within 30 minutes",
        "urgent",
        "act fast",
        "last warning",
        "final notice",
        "before it is too late",
    ],
    "isolation_tactics": [
        "do not disconnect",
        "don't disconnect",
        "do not tell anyone",
        "keep this confidential",
        "do not hang up",
        "stay on the call",
        "do not inform your family",
    ],
    "payment_request": [
        "transfer",
        "pay a fine",
        "processing fee",
        "refundable deposit",
        "security deposit",
        "verification fee",
        "send money",
        "pay now",
    ],
    "criminal_allegation": [
        "money laundering",
        "your aadhaar is linked",
        "parcel contains drugs",
        "illegal activity",
        "your account is involved",
        "under investigation",
    ],
}

In [19]:
def _extract_entities(text: str) -> Dict[str, List[str]]:
    """Run every regex pattern against the text and collect unique matches."""
    entities: Dict[str, List[str]] = {}
    for label, pattern in _PATTERNS.items():
        found = re.findall(pattern, text, flags=re.IGNORECASE)
        seen: List[str] = []
        for item in found:
            cleaned = item.strip()
            if cleaned and cleaned not in seen:
                seen.append(cleaned)
        if seen:
            entities[label] = seen
    return entities

In [20]:
def _extract_behaviours(
    text: str,
) -> Tuple[Dict[str, bool], Dict[str, List[str]]]:
    """Detect behavioural / linguistic signals via keyword matching."""
    lowered = text.lower()
    behaviours: Dict[str, bool] = {}
    matches: Dict[str, List[str]] = {}

    for behaviour_name, keywords in _BEHAVIOUR_KEYWORD_GROUPS.items():
        found_terms = [kw for kw in keywords if kw in lowered]
        behaviours[behaviour_name] = len(found_terms) > 0
        if found_terms:
            matches[behaviour_name] = found_terms

    return behaviours, matches

In [21]:
def extract_signals(text: str) -> SignalExtractionResult:
    """
    Module 1 entry point.

    Extracts entities and behavioural signals from raw user-reported text.
    Does not classify the case. Raises ValueError on empty input.
    """
    if not text or not text.strip():
        raise ValueError("extract_signals received empty input text.")

    entities = _extract_entities(text)
    behaviours, matches = _extract_behaviours(text)

    result = SignalExtractionResult(
        entities=entities, behaviours=behaviours, raw_matches=matches
    )
    logger.info(
        "Signal extraction complete. entities=%d behaviour_flags=%d",
        len(entities),
        sum(1 for v in behaviours.values() if v),
    )
    return result

## 4. Module 2 - Knowledge Retrieval (RAG)

In [22]:
_embedding_model: Optional[SentenceTransformer] = None
_chroma_collection = None


def _get_embedding_model() -> SentenceTransformer:
    """Lazily load the sentence-transformers model used in Notebook 1."""
    global _embedding_model
    if _embedding_model is None:
        logger.info("Loading embedding model: %s", CONFIG.EMBEDDING_MODEL_NAME)
        _embedding_model = SentenceTransformer(CONFIG.EMBEDDING_MODEL_NAME)
    return _embedding_model

In [23]:
def _get_chroma_collection():
    """
    Connect to the existing ChromaDB collection created in Notebook 1.

    Raises RuntimeError if the collection does not exist.
    """
    global _chroma_collection
    if _chroma_collection is None:
        client = chromadb.PersistentClient(path=CONFIG.CHROMA_PERSIST_DIR)
        existing_names = [c.name for c in client.list_collections()]
        if CONFIG.CHROMA_COLLECTION_NAME not in existing_names:
            raise RuntimeError(
                f"ChromaDB collection '{CONFIG.CHROMA_COLLECTION_NAME}' "
                f"was not found at '{CONFIG.CHROMA_PERSIST_DIR}'. "
                f"Collections found there: {existing_names}. "
                f"Update CONFIG.CHROMA_COLLECTION_NAME to match Notebook 1, "
                f"or run Notebook 1 first if that list is empty."
            )
        _chroma_collection = client.get_collection(CONFIG.CHROMA_COLLECTION_NAME)
        logger.info(
            "Connected to ChromaDB collection '%s' (%d documents).",
            CONFIG.CHROMA_COLLECTION_NAME,
            _chroma_collection.count(),
        )
    return _chroma_collection

In [24]:
@dataclass
class RetrievedChunk:
    """One retrieved knowledge base chunk with its metadata."""

    text: str
    source_document: str
    category: str
    metadata: Dict[str, Any]
    relevance_score: float

In [25]:
def _resolve_source_document(meta: Dict[str, Any]) -> str:
    """
    Resolve the source document name across possible metadata key names.
    """
    return (
        meta.get("source_file")
        or meta.get("source_document")
        or meta.get("doc_id")
        or "unknown"
    )

In [26]:
def _l2_distance_to_similarity(distance: float) -> float:
    """
    Convert a ChromaDB L2 distance to a 0-1 similarity score.

    ChromaDB's default metric for PersistentClient is L2 (squared Euclidean).
    Values range from 0 (identical vectors) upward with no fixed maximum.

    Formula: similarity = 1 / (1 + distance)
    - distance = 0   -> similarity = 1.0
    - distance = 1   -> similarity = 0.5
    - distance = 9   -> similarity = 0.1

    This is monotone-decreasing, so ranking is preserved.
    """
    return 1.0 / (1.0 + float(distance))

In [27]:
def retrieve_evidence(query: str, top_k: int = None) -> List[RetrievedChunk]:
    """
    Module 2 entry point.

    Embeds the query with the same model used in Notebook 1, queries
    ChromaDB for the top-k most relevant chunks, and returns them with
    their source metadata. Filters out chunks below MIN_RELEVANCE_SCORE.
    """
    if not query or not query.strip():
        raise ValueError("retrieve_evidence received empty query text.")

    top_k = top_k or CONFIG.TOP_K
    model = _get_embedding_model()
    collection = _get_chroma_collection()

    query_embedding = model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    chunks: List[RetrievedChunk] = []
    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    for doc_text, meta, distance in zip(documents, metadatas, distances):
        meta = meta or {}
        relevance_score = _l2_distance_to_similarity(distance)

        if relevance_score < CONFIG.MIN_RELEVANCE_SCORE:
            continue

        chunks.append(
            RetrievedChunk(
                text=doc_text,
                source_document=_resolve_source_document(meta),
                category=meta.get("category", "uncategorised"),
                metadata=meta,
                relevance_score=round(relevance_score, 4),
            )
        )

    logger.info(
        "Retrieved %d relevant chunks (of %d candidates) for query.",
        len(chunks),
        len(documents),
    )
    return chunks

## 5. Module 3 - LLM Reasoning (OpenRouter / tencent/hy3:free)

In [28]:
def _get_openrouter_api_key() -> str:
    """
    Return the OpenRouter API key from the environment.

    FIX-2026-07-10: no hardcoded fallback key anymore. A real key was
    previously embedded directly in this file - that key should be
    treated as compromised and rotated on openrouter.ai immediately.
    This function now fails loudly instead of silently using a leaked
    key, so the mistake can't happen again.
    """
    api_key = os.environ.get(CONFIG.OPENROUTER_API_KEY_ENV_VAR)
    if not api_key:
        raise RuntimeError(
            f"Environment variable '{CONFIG.OPENROUTER_API_KEY_ENV_VAR}' is not "
            f"set. Set it before running this notebook, e.g.:\n"
            f"  import os\n"
            f"  os.environ['{CONFIG.OPENROUTER_API_KEY_ENV_VAR}'] = '<your key>'\n"
            f"Never hardcode API keys directly in notebook/source files."
        )
    return api_key

In [29]:
_REASONING_SYSTEM_INSTRUCTIONS = """\
You are a fraud intelligence reasoning engine for a Digital Public Safety
platform. You analyse a citizen-reported conversation using only the
extracted signals and the retrieved official evidence provided to you.

Rules you must follow strictly:
1. Reason only from the evidence given to you. Do not invent facts.
2. If the retrieved evidence is insufficient to support a conclusion,
   explicitly say so instead of guessing.
3. Identify fraud indicators as short labels (for example
   'Government Impersonation', 'Threat Language', 'Urgent Payment Request').
   Only use 'Government Impersonation' when the message explicitly claims
   to be from a specific government agency, police force, or official body
   (e.g. "I am from CBI", "this is the Income Tax Department"). Do NOT use
   it just because the message mentions arrest, criminal case, or legal
   action in general - use 'Threat Language' or 'Criminal Allegation' for
   that instead.
   Do NOT decide the overall fraud type or category yourself.
   Do NOT output a numeric risk score.
   Both are computed by deterministic logic downstream from your indicators.
4. Do not recommend any action to the citizen, bank, police or telecom
   operator. Your only job is to explain what is happening.
5. Keep the "explanation" field to at most 3 sentences. Be concise -
   your entire response must fit comfortably within the output budget.
6. Your ENTIRE response must be a single raw JSON object - no markdown fences,
   no prose before or after it - with exactly these four keys:
     "indicators"         : array of short indicator strings
     "llm_confidence"     : integer 0-100
     "evidence_sufficient": boolean
     "explanation"        : short paragraph grounded in the evidence
"""

In [30]:
def _build_reasoning_prompt(
    user_input: str,
    signals: SignalExtractionResult,
    retrieved_chunks: List[RetrievedChunk],
) -> str:
    """Assemble the user-turn prompt combining input, signals, and evidence."""
    signals_block = json.dumps(
        {"entities": signals.entities, "behaviours": signals.behaviours},
        indent=2,
    )

    if retrieved_chunks:
        evidence_block = "\n\n".join(
            f"[Source: {c.source_document} | Category: {c.category} | "
            f"Relevance: {c.relevance_score}]\n{c.text}"
            for c in retrieved_chunks
        )
    else:
        evidence_block = (
            "No relevant documents were retrieved from the knowledge base."
        )

    return (
        f"Citizen-reported conversation:\n{json.dumps(user_input)}\n\n"
        f"Extracted signals:\n{signals_block}\n\n"
        f"Retrieved official evidence:\n{evidence_block}\n\n"
        f"Based only on the above, return the JSON object described in "
        f"your instructions. Output ONLY the raw JSON - no markdown, no prose."
    )

In [31]:
def _extract_json_from_text(text: str) -> str:
    """
    Best-effort extraction of the first complete JSON object from arbitrary
    text. Handles three cases in priority order:

    1. The whole text (after stripping fences) is already valid JSON.
    2. There is a {...} block somewhere in the text.
    3. The text contains a JSON object spread across multiple lines.
    """
    text = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text.strip())
    text = text.strip()

    if text.startswith("{"):
        return text

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        return match.group(0)

    return text

In [32]:
def _repair_truncated_json(text: str) -> str:
    """
    FIX-2026-07-10: last-resort repair for a JSON object that was cut off
    mid-generation (finish_reason == "length"). This is a heuristic, not
    a full parser - it handles the common truncation shapes we actually
    saw (cut off mid-string, or mid-array/object) well enough to recover
    the fields that were already fully written.

    Strategy:
    1. If we're mid-string (odd number of unescaped double quotes), close
       the string.
    2. Trim any dangling trailing comma.
    3. Close any open arrays/objects in the correct order based on a
       simple bracket-depth scan.
    """
    repaired = text

    # Count unescaped double quotes to detect an unterminated string.
    unescaped_quotes = len(re.findall(r'(?<!\\)"', repaired))
    if unescaped_quotes % 2 == 1:
        repaired += '"'

    # Drop a trailing comma (with optional whitespace) right before EOF.
    repaired = re.sub(r",\s*$", "", repaired)

    # Walk the string tracking bracket depth (ignoring bracket chars that
    # occur inside string literals) to build the correct closing sequence.
    stack: List[str] = []
    in_string = False
    escape_next = False
    for ch in repaired:
        if escape_next:
            escape_next = False
            continue
        if ch == "\\":
            escape_next = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch in "{[":
            stack.append(ch)
        elif ch in "}]":
            if stack:
                stack.pop()

    closers = {"{": "}", "[": "]"}
    while stack:
        opener = stack.pop()
        repaired += closers[opener]

    return repaired

In [33]:
def _extract_text_from_response(response_json: Dict[str, Any]) -> str:
    """
    Extract the model's final answer text from an OpenRouter response.

    tencent/hy3:free is a reasoning model and may return its answer in
    different places depending on whether it used extended thinking:

    Shape A - normal:
        choices[0].message.content = "<the answer>"
        choices[0].message.reasoning_details = [...]   (chain-of-thought)

    Shape B - reasoning-only (content is None or ""):
        choices[0].message.content = None / ""
        choices[0].message.reasoning_details = [
            {"type": "thinking", "thinking": "...chain of thought..."},
            {"type": "text",     "text":     "<the answer>"}   <- last item
        ]

    Shape C - reasoning_details absent, everything in content:
        choices[0].message.content = "<the answer>"

    This function tries each shape in order and returns the first non-empty
    string it finds.
    """
    message = response_json["choices"][0]["message"]
    content: str = (message.get("content") or "").strip()
    reasoning_details: List[Dict] = message.get("reasoning_details") or []

    if content:
        logger.debug("Response shape: content field is populated (Shape A/C).")
        return content

    if reasoning_details:
        text_blocks = [
            item.get("text", "")
            for item in reasoning_details
            if isinstance(item, dict) and item.get("type") == "text"
        ]
        if text_blocks:
            logger.debug(
                "Response shape: answer found in reasoning_details 'text' block "
                "(Shape B). %d text block(s) found, using last.",
                len(text_blocks),
            )
            return text_blocks[-1].strip()

        thinking_blocks = [
            item.get("thinking", "")
            for item in reasoning_details
            if isinstance(item, dict) and item.get("type") == "thinking"
        ]
        if thinking_blocks:
            combined = "\n".join(thinking_blocks)
            logger.debug(
                "Response shape: no 'text' block; falling back to 'thinking' "
                "content (%d chars).",
                len(combined),
            )
            return combined

    logger.warning(
        "Could not extract any text from OpenRouter response: %s",
        json.dumps(response_json, default=str)[:500],
    )
    return ""

In [34]:
def _call_openrouter(
    messages: List[Dict[str, Any]], max_output_tokens: int
) -> Dict[str, Any]:
    """
    POST to OpenRouter and return the parsed response JSON.
    Raises RuntimeError on HTTP errors or malformed responses.
    """
    api_key = _get_openrouter_api_key()
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": CONFIG.OPENROUTER_MODEL_NAME,
        "messages": messages,
        "temperature": CONFIG.LLM_TEMPERATURE,
        "max_tokens": max_output_tokens,
        # FIX-2026-07-10: cap the hidden reasoning budget so it can't
        # consume the whole max_tokens allowance and starve the final
        # JSON answer, which is what caused the truncation crash.
        "reasoning": {"max_tokens": CONFIG.LLM_REASONING_MAX_TOKENS},
        # FIX-2026-07-11: removed "response_format": {"type": "json_object"}.
        # The Novita provider backing tencent/hy3:free rejects it outright
        # ("does not support 'json_object' response format... supported
        # formats: json_schema"), causing an immediate 400 on every call.
        # We rely instead on the system prompt instruction plus
        # _extract_json_from_text / _repair_truncated_json downstream,
        # which already handle malformed/truncated output robustly.
    }

    response = requests.post(
        CONFIG.OPENROUTER_API_URL,
        headers=headers,
        data=json.dumps(payload),
        timeout=180,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"OpenRouter API request failed with status {response.status_code}: "
            f"{response.text}"
        )

    parsed = response.json()

    if "choices" not in parsed or not parsed["choices"]:
        raise RuntimeError(
            f"OpenRouter response contained no choices: {json.dumps(parsed)}"
        )

    return parsed

In [35]:
def run_llm_reasoning(
    user_input: str,
    signals: SignalExtractionResult,
    retrieved_chunks: List[RetrievedChunk],
) -> Dict[str, Any]:
    """
    Module 3 entry point.

    Single-turn call to tencent/hy3:free via OpenRouter with a bounded
    reasoning budget. If the response comes back truncated, retries once
    with a larger token budget; if it's still truncated, attempts a
    best-effort repair rather than failing the whole case.
    """
    user_prompt = _build_reasoning_prompt(user_input, signals, retrieved_chunks)

    messages = [
        {"role": "system", "content": _REASONING_SYSTEM_INSTRUCTIONS},
        {"role": "user", "content": user_prompt},
    ]

    token_budget = CONFIG.LLM_MAX_OUTPUT_TOKENS
    last_raw_text = ""
    was_truncated = False

    for attempt in range(CONFIG.LLM_MAX_RETRIES + 1):
        logger.info(
            "Sending request to OpenRouter (%s), attempt %d, max_tokens=%d.",
            CONFIG.OPENROUTER_MODEL_NAME,
            attempt + 1,
            token_budget,
        )
        response_json = _call_openrouter(messages, token_budget)

        finish_reason = (
            response_json.get("choices", [{}])[0].get("finish_reason", "")
        )
        was_truncated = finish_reason == "length"

        raw_text = _extract_text_from_response(response_json)
        last_raw_text = raw_text or last_raw_text

        if not raw_text:
            logger.error(
                "Model returned completely empty text. Full response: %s",
                json.dumps(response_json, default=str)[:2000],
            )
            if attempt < CONFIG.LLM_MAX_RETRIES:
                token_budget = CONFIG.LLM_RETRY_MAX_OUTPUT_TOKENS
                time.sleep(1)
                continue
            raise ValueError(
                "Model returned an empty response after retries. Check the "
                "full response logged above."
            )

        cleaned = _extract_json_from_text(raw_text)

        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError as exc:
            if was_truncated and attempt < CONFIG.LLM_MAX_RETRIES:
                logger.warning(
                    "Response truncated (finish_reason=length) and unparseable; "
                    "retrying with a larger token budget."
                )
                token_budget = CONFIG.LLM_RETRY_MAX_OUTPUT_TOKENS
                time.sleep(1)
                continue

            # Last resort: attempt heuristic repair instead of failing outright.
            logger.warning(
                "JSON parse failed (%s). Attempting truncation repair.", exc
            )
            repaired = _repair_truncated_json(cleaned)
            try:
                parsed = json.loads(repaired)
                logger.warning(
                    "Truncation repair succeeded; result may be partial."
                )
            except json.JSONDecodeError as exc2:
                logger.error(
                    "Could not parse model output as JSON even after repair.\n"
                    "Raw text: %s\nCleaned: %s\nRepaired: %s",
                    raw_text[:1000],
                    cleaned[:1000],
                    repaired[:1000],
                )
                raise ValueError(
                    f"Failed to parse LLM output as JSON: {exc2}"
                ) from exc2

        # Fill in any missing required keys with safe defaults rather than
        # failing the whole case - this matters most right after a repair.
        required_defaults = {
            "indicators": [],
            "llm_confidence": 0,
            "evidence_sufficient": False,
            "explanation": "[Response was truncated; partial data shown.]",
        }
        missing = set(required_defaults) - parsed.keys()
        if missing:
            logger.warning("LLM JSON response missing keys %s; using defaults.", missing)
            for key in missing:
                parsed[key] = required_defaults[key]

        logger.info(
            "LLM reasoning complete. indicators=%d evidence_sufficient=%s",
            len(parsed["indicators"]),
            parsed["evidence_sufficient"],
        )
        return parsed

    # Should not be reachable, but keep a defined failure path.
    raise ValueError(
        f"Failed to obtain a usable LLM response. Last raw text: {last_raw_text[:500]}"
    )

## 6. Module 3b - Indicator Normalisation

In [36]:
_CANONICAL_INDICATOR_SYNONYMS: Dict[str, List[str]] = {
    "Government Impersonation": [
        "government impersonation",
        "government official impersonation",
        "authority impersonation",
        "police impersonation",
        "cbi impersonation",
        "impersonating a government agency",
        "impersonation of bank official",
        "official impersonation",
    ],
    "Threat Language": [
        "threat language",
        "fear tactics",
        "intimidation",
        "legal threat",
        "threatening language",
        "coercion",
    ],
    "Urgency": [
        "urgency",
        "time pressure",
        "urgent request",
        "artificial urgency",
    ],
    "Isolation Tactics": [
        "isolation",
        "isolation tactics",
        "do not disconnect instruction",
        "keep this confidential instruction",
    ],
    "Money Demand": [
        "money demand",
        "urgent transfer request",
        "payment demand",
        "fund transfer request",
        "demand for money",
        "financial demand",
    ],
    "Urgent Payment Request": [
        "urgent payment request",
        "immediate payment demand",
        "request for immediate transfer",
    ],
    "Criminal Allegation": [
        "criminal allegation",
        "false criminal accusation",
        "accusation of a crime",
        "false legal accusation",
    ],
    "Suspicious Link": [
        "suspicious link",
        "phishing link",
        "malicious url",
        "fake link",
    ],
    "OTP Request": [
        "otp request",
        "otp phishing",
        "request for otp",
        "asking for otp",
    ],
    "Blackmail / Extortion Threat": [
        "blackmail",
        "extortion",
        "extortion threat",
        "sextortion",
        "private photo blackmail",
        "photo leak threat",
        "leak threat",
        "threat to leak",
        "threat to expose",
        "compromising photo threat",
        "compromising video threat",
    ],
}

_SYNONYM_TO_CANONICAL: Dict[str, str] = {
    synonym.lower(): canonical
    for canonical, synonyms in _CANONICAL_INDICATOR_SYNONYMS.items()
    for synonym in synonyms + [canonical]
}
_ALL_SYNONYMS_LOWER: List[str] = list(_SYNONYM_TO_CANONICAL.keys())

In [37]:
def normalize_indicator(raw_indicator: str) -> str:
    """
    Map a raw indicator string from the LLM onto a canonical indicator
    label. Falls back to fuzzy matching, then to the original string if no
    sufficiently close match is found.
    """
    cleaned = raw_indicator.strip().lower()
    if not cleaned:
        return raw_indicator

    if cleaned in _SYNONYM_TO_CANONICAL:
        return _SYNONYM_TO_CANONICAL[cleaned]

    close_matches = difflib.get_close_matches(
        cleaned,
        _ALL_SYNONYMS_LOWER,
        n=1,
        cutoff=CONFIG.INDICATOR_FUZZY_MATCH_CUTOFF,
    )
    if close_matches:
        return _SYNONYM_TO_CANONICAL[close_matches[0]]

    return raw_indicator.strip()

In [38]:
def normalize_indicators(raw_indicators: List[str]) -> List[str]:
    """
    Normalise a list of raw indicators and de-duplicate the result while
    preserving order.
    """
    normalized: List[str] = []
    seen: set = set()
    for raw in raw_indicators:
        canonical = normalize_indicator(raw)
        key = canonical.lower()
        if key not in seen:
            seen.add(key)
            normalized.append(canonical)
    return normalized

## 7. Module 4 - Fraud Classification (Deterministic)

In [39]:
_FRAUD_TYPE_SIGNATURES: Dict[str, List[str]] = {
    "Digital Arrest Scam": [
        "Government Impersonation",
        "Threat Language",
        "Isolation Tactics",
        "Criminal Allegation",
        "Urgent Payment Request",
    ],
    "UPI / Payment Fraud": [
        "Money Demand",
        "Urgent Payment Request",
        "Suspicious Link",
        "OTP Request",
    ],
    "Phishing / Credential Theft": [
        "Suspicious Link",
        "OTP Request",
        "Urgency",
    ],
    "Extortion / Threat-based Fraud": [
        "Threat Language",
        "Criminal Allegation",
        "Money Demand",
        "Blackmail / Extortion Threat",
    ],
}

In [40]:
def _overlap_ratio(case_indicators: set, signature_indicators: List[str]) -> float:
    """Fraction of a signature's indicators that are present in the case."""
    if not signature_indicators:
        return 0.0
    matched = case_indicators & set(signature_indicators)
    return len(matched) / len(signature_indicators)

In [41]:
def classify_fraud_type(normalized_indicators: List[str]) -> Dict[str, Any]:
    """
    Module 4 entry point.

    Deterministically maps normalised indicators to the best-matching
    fraud type using fixed signatures.
    """
    if not normalized_indicators:
        return {
            "fraud_type": CONFIG.UNCLASSIFIED_LABEL,
            "match_ratio": 0.0,
            "candidates": {},
        }

    case_set = set(normalized_indicators)
    scores: Dict[str, float] = {
        fraud_type: _overlap_ratio(case_set, signature)
        for fraud_type, signature in _FRAUD_TYPE_SIGNATURES.items()
    }

    best_fraud_type = max(scores, key=scores.get)
    best_score = scores[best_fraud_type]

    chosen = (
        best_fraud_type
        if best_score >= CONFIG.CLASSIFICATION_MIN_OVERLAP_RATIO
        else CONFIG.UNCLASSIFIED_LABEL
    )

    logger.info(
        "Fraud classification complete. chosen=%s best_ratio=%.2f",
        chosen,
        best_score,
    )
    return {
        "fraud_type": chosen,
        "match_ratio": round(best_score, 4),
        "candidates": {k: round(v, 4) for k, v in scores.items()},
    }

## 8. Module 5 - Evidence Validation

In [42]:
@dataclass
class ValidationResult:
    """Output of Module 5, feeding into the final confidence score."""

    adjusted_confidence: int
    validation_notes: str
    category_match_ratio: float

In [43]:
def _category_relates_to_fraud_type(category: str, fraud_type: str) -> bool:
    """
    Loose relatedness check between a retrieved chunk's category and the
    classified fraud type.
    """

    def _normalize(s: str) -> set:
        return set(re.findall(r"[a-z0-9]+", s.lower()))

    category_tokens = _normalize(category)
    fraud_type_tokens = _normalize(fraud_type)
    if not category_tokens or not fraud_type_tokens:
        return False
    return len(category_tokens & fraud_type_tokens) > 0

In [44]:
def validate_evidence(
    llm_output: Dict[str, Any],
    classification: Dict[str, Any],
    retrieved_chunks: List[RetrievedChunk],
) -> ValidationResult:
    """
    Module 5 entry point.

    Adjusts the LLM's self-reported confidence based on retrieval strength,
    category relatedness to the classified fraud type, and whether the LLM
    itself flagged the evidence as insufficient.
    """
    base_confidence = int(llm_output.get("llm_confidence", 0))
    evidence_sufficient = bool(llm_output.get("evidence_sufficient", False))
    fraud_type = classification.get("fraud_type", CONFIG.UNCLASSIFIED_LABEL)

    if not retrieved_chunks:
        return ValidationResult(
            adjusted_confidence=max(0, base_confidence - 40),
            validation_notes=(
                "No supporting documents were retrieved; confidence reduced sharply."
            ),
            category_match_ratio=0.0,
        )

    if not evidence_sufficient:
        return ValidationResult(
            adjusted_confidence=max(0, base_confidence - 25),
            validation_notes=(
                "The model flagged retrieved evidence as insufficient; "
                "confidence reduced."
            ),
            category_match_ratio=0.0,
        )

    related_flags = [
        _category_relates_to_fraud_type(c.category, fraud_type)
        for c in retrieved_chunks
    ]
    category_match_ratio = sum(related_flags) / len(related_flags)
    avg_relevance = (
        sum(c.relevance_score for c in retrieved_chunks) / len(retrieved_chunks)
    )

    if category_match_ratio >= 0.5 and avg_relevance >= 0.6:
        adjusted = min(100, base_confidence + 15)
        notes = (
            "Retrieved documents are both highly relevant and match the "
            "classified fraud type's category; confidence increased."
        )
    elif category_match_ratio >= 0.5:
        adjusted = min(100, base_confidence + 5)
        notes = (
            "Retrieved documents match the classified fraud type's category, "
            "with moderate topical relevance; confidence slightly increased."
        )
    elif avg_relevance >= 0.75:
        adjusted = base_confidence
        notes = (
            "Retrieved documents are highly relevant to the query but not "
            "clearly categorised under the classified fraud type; "
            "confidence unchanged."
        )
    else:
        adjusted = max(0, base_confidence - 20)
        notes = (
            "Retrieved documents are only weakly relevant and do not match "
            "the classified fraud type's category; confidence reduced."
        )

    return ValidationResult(
        adjusted_confidence=adjusted,
        validation_notes=notes,
        category_match_ratio=round(category_match_ratio, 4),
    )

## 9. Module 6 - Rule-based Risk Engine

In [45]:
class Severity(str, Enum):
    LOW = "Low"
    MEDIUM = "Medium"
    HIGH = "High"
    CRITICAL = "Critical"

In [46]:
_INDICATOR_WEIGHTS: Dict[str, int] = {
    "Government Impersonation": 25,
    "Threat Language": 20,
    "Urgency": 15,
    "Urgent Payment Request": 15,
    "Money Demand": 30,
    "Isolation Tactics": 10,
    "Criminal Allegation": 20,
    "Suspicious Link": 10,
    "OTP Request": 20,
    "Blackmail / Extortion Threat": 25,
}
_DEFAULT_INDICATOR_WEIGHT = 8

In [47]:
def _severity_from_score(score: int) -> Severity:
    if score <= CONFIG.RISK_LOW_MAX:
        return Severity.LOW
    if score <= CONFIG.RISK_MEDIUM_MAX:
        return Severity.MEDIUM
    if score <= CONFIG.RISK_HIGH_MAX:
        return Severity.HIGH
    return Severity.CRITICAL

In [48]:
def compute_risk_score(indicators: List[str]) -> Dict[str, Any]:
    """
    Module 6 entry point.

    Converts a list of indicator labels into a deterministic risk score
    (0-100, capped) and a severity band. De-duplicates indicators before scoring.
    """
    if not indicators:
        return {"risk_score": 0, "severity": Severity.LOW.value, "breakdown": {}}

    deduped: List[str] = []
    seen: set = set()
    for indicator in indicators:
        key = indicator.strip().lower()
        if key not in seen:
            seen.add(key)
            deduped.append(indicator.strip())

    breakdown: Dict[str, int] = {}
    total = 0
    for indicator in deduped:
        weight = _INDICATOR_WEIGHTS.get(indicator, _DEFAULT_INDICATOR_WEIGHT)
        breakdown[indicator] = weight
        total += weight

    capped_score = min(100, total)
    return {
        "risk_score": capped_score,
        "severity": _severity_from_score(capped_score).value,
        "breakdown": breakdown,
    }

## 10. Orchestration - Full Pipeline

In [49]:
class FraudIntelligenceError(Exception):
    """Raised when the pipeline cannot produce a valid intelligence result."""

In [50]:
def analyze_case(user_input: str) -> Dict[str, Any]:
    """
    Full Notebook 2 pipeline entry point.

    Runs all modules in sequence and returns the final structured
    Fraud Intelligence JSON as a Python dictionary.
    """
    case_id = str(uuid.uuid4())
    timestamp = datetime.now(timezone.utc).isoformat()

    try:
        signals = extract_signals(user_input)
        retrieved_chunks = retrieve_evidence(user_input, top_k=CONFIG.TOP_K)
        llm_output = run_llm_reasoning(user_input, signals, retrieved_chunks)

        normalized_indicators = normalize_indicators(
            llm_output.get("indicators", [])
        )
        classification = classify_fraud_type(normalized_indicators)
        validation = validate_evidence(llm_output, classification, retrieved_chunks)
        risk = compute_risk_score(normalized_indicators)

        result = {
            "case_id": case_id,
            "timestamp": timestamp,
            "fraud_type": classification["fraud_type"],
            "classification_match_ratio": classification["match_ratio"],
            "confidence": validation.adjusted_confidence,
            "severity": risk["severity"],
            "risk_score": risk["risk_score"],
            "risk_breakdown": risk["breakdown"],
            "indicators": normalized_indicators,
            "entities": signals.entities,
            "behavioural_signals": {
                k: v for k, v in signals.behaviours.items() if v
            },
            "retrieved_documents": [
                {
                    "source": c.source_document,
                    "category": c.category,
                    "relevance_score": c.relevance_score,
                }
                for c in retrieved_chunks
            ],
            "evidence_sufficient": llm_output.get("evidence_sufficient", False),
            "category_match_ratio": validation.category_match_ratio,
            "validation_notes": validation.validation_notes,
            "summary": llm_output.get("explanation", ""),
        }

        logger.info(
            "Case %s analysed. fraud_type=%s severity=%s confidence=%d",
            case_id,
            result["fraud_type"],
            result["severity"],
            result["confidence"],
        )
        return result

    except Exception as exc:
        logger.exception("Failed to analyse case %s", case_id)
        raise FraudIntelligenceError(
            f"Pipeline failed for case {case_id}: {exc}"
        ) from exc

## 11. Test Cases

Before running this cell, set your OpenRouter API key, for example in a
previous cell:

```python
import os
os.environ["OPENROUTER_API_KEY"] = "sk-or-v1-..."
```

Never hardcode the key directly in a committed notebook cell.

In [51]:
SAMPLE_CASES = [
    (
        "digital_arrest_case",
        "Hello Sir, I am from CBI. Your Aadhaar is linked with money "
        "laundering. Do not disconnect the call. Transfer Rs 2,00,000 "
        "immediately to avoid arrest.",
    ),
    (
        "benign_case",
        "Hi, just checking if you are free for lunch tomorrow at 1pm.",
    ),
]

In [52]:
# Set your API key BEFORE running, e.g.:
#   os.environ["OPENROUTER_API_KEY"] = "sk-or-v1-..."
# (do this in a separate, non-committed cell/file - never hardcode it here)

test_results: Dict[str, Any] = {}
for case_name, case_text in SAMPLE_CASES:
    try:
        test_results[case_name] = analyze_case(case_text)
    except FraudIntelligenceError as exc:
        logger.error("Test case '%s' failed: %s", case_name, exc)
        test_results[case_name] = {"error": str(exc)}

for case_name, output in test_results.items():
    print(f"--- {case_name} ---")
    print(json.dumps(output, indent=2, default=str))
    print()

2026-07-11 06:16:30,756 | INFO | fraud_intelligence_engine | Signal extraction complete. entities=1 behaviour_flags=5
2026-07-11 06:16:30,760 | INFO | fraud_intelligence_engine | Loading embedding model: all-MiniLM-L6-v2
2026-07-11 06:16:31,831 | INFO | sentence_transformers.base.model | No device provided, using cpu
2026-07-11 06:16:35,569 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-11 06:16:35,607 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-07-11 06:16:35,914 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-11 06:16:35,937 | INFO | httpx | HTTP Request: HEAD https://huggingfa

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-07-11 06:16:42,669 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-11 06:16:42,905 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-11 06:16:43,181 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-11 06:16:43,400 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-11 06:16:43,703 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-11 06:16:43,732 | INFO | httpx | HTTP Request: HEAD https:

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-11 06:16:48,168 | INFO | fraud_intelligence_engine | Retrieved 5 relevant chunks (of 5 candidates) for query.
2026-07-11 06:16:48,178 | INFO | fraud_intelligence_engine | Sending request to OpenRouter (tencent/hy3:free), attempt 1, max_tokens=4096.
2026-07-11 06:17:05,609 | INFO | fraud_intelligence_engine | LLM reasoning complete. indicators=4 evidence_sufficient=True
2026-07-11 06:17:05,609 | INFO | fraud_intelligence_engine | Fraud classification complete. chosen=Digital Arrest Scam best_ratio=0.80
2026-07-11 06:17:05,609 | INFO | fraud_intelligence_engine | Case c0f870f5-d688-4db7-b9a9-73b9136ec3d4 analysed. fraud_type=Digital Arrest Scam severity=High confidence=75
2026-07-11 06:17:05,622 | INFO | fraud_intelligence_engine | Signal extraction complete. entities=0 behaviour_flags=0


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-11 06:17:05,795 | INFO | fraud_intelligence_engine | Retrieved 5 relevant chunks (of 5 candidates) for query.
2026-07-11 06:17:05,800 | INFO | fraud_intelligence_engine | Sending request to OpenRouter (tencent/hy3:free), attempt 1, max_tokens=4096.
2026-07-11 06:17:26,126 | INFO | fraud_intelligence_engine | LLM reasoning complete. indicators=0 evidence_sufficient=False
2026-07-11 06:17:26,126 | INFO | fraud_intelligence_engine | Case 8d5f355f-0de2-4616-b087-59a9d26567a5 analysed. fraud_type=Unclassified Suspicious Activity severity=Low confidence=65


--- digital_arrest_case ---
{
  "case_id": "c0f870f5-d688-4db7-b9a9-73b9136ec3d4",
  "timestamp": "2026-07-11T00:46:30.756500+00:00",
  "fraud_type": "Digital Arrest Scam",
  "classification_match_ratio": 0.8,
  "confidence": 75,
  "severity": "High",
  "risk_score": 70,
  "risk_breakdown": {
    "Government Impersonation": 25,
    "Urgent Payment Request": 15,
    "Isolation Tactics": 10,
    "Criminal Allegation": 20
  },
  "indicators": [
    "Government Impersonation",
    "Urgent Payment Request",
    "Isolation Tactics",
    "Criminal Allegation"
  ],
  "entities": {
    "money_amounts": [
      "Rs 2,00,000"
    ]
  },
  "behavioural_signals": {
    "authority_impersonation": true,
    "urgency": true,
    "isolation_tactics": true,
    "payment_request": true,
    "criminal_allegation": true
  },
  "retrieved_documents": [
    {
      "source": "ADVISORYTAU-ADV-003DigitalArrest06.03.2025.pdf",
      "category": "digital_arrest",
      "relevance_score": 0.5623
    },
    {
    

## 12. Notes and Limitations

- SECURITY FIX: removed a hardcoded real API key that was previously
  embedded in both `Config.OPENROUTER_API_KEY_ENV_VAR` and a fallback
  constant. That key must be rotated on openrouter.ai - it should be
  treated as already leaked. The key now must come from the
  `OPENROUTER_API_KEY` environment variable, with a loud failure if unset.
- FIXED: JSON truncation crash - tencent/hy3:free's hidden reasoning
  was consuming most of `max_tokens=2048`, cutting the final JSON off
  mid-string. Fixed by (a) capping reasoning tokens separately via
  `reasoning.max_tokens`, (b) raising the overall `max_tokens` budget,
  (c) detecting `finish_reason == "length"` and retrying once with a
  larger budget, and (d) a heuristic repair pass as a last resort so a
  partially truncated answer still produces a usable (flagged) result
  instead of crashing the whole case.
- FIXED (2026-07-11): `"response_format": {"type": "json_object"}` was
  briefly added to force valid JSON, but the Novita provider backing
  tencent/hy3:free rejects that format with a 400 (it only supports
  `"json_schema"`). Removed - the system prompt instruction plus the
  parse/repair pipeline below already gets valid JSON reliably without it.
- FIXED: L2 distance to similarity conversion - uses `1/(1+distance)`.
- FIXED: response shape handling for tencent/hy3:free's `reasoning_details`.
- The fraud type signature table (Module 4) and indicator weight table
  (Module 6) are starting points for the hackathon demo.
- Indicator normalisation uses a fixed synonym list plus fuzzy matching.
- This notebook produces intelligence only; action recommendations belong
  in Notebook 3.
- Regex-based entity extraction is intentionally simple for the MVP.

## 13. Deterministic Test Suite

Originally provided as a separate script, `test_fraud_intelligence_engine.py`,
meant to be run after importing everything above and after setting
`os.environ["OPENROUTER_API_KEY"]`.

This does NOT call any AI to grade itself - it is a plain deterministic
comparator. Since Module 3 (LLM reasoning) is non-deterministic, indicator
lists are compared as "expected indicators are a subset of what came back"
rather than exact match - that is the realistic bar for a live LLM call.

In [53]:
import json
from typing import Any, Dict, List

In [54]:
# ---------------------------------------------------------------------------
# Test cases: (name, input_text, expected_dict)
#
# expected_dict fields:
#   fraud_type          - exact string match required
#   severity            - exact string match required
#   min_indicators      - normalized indicators must be a superset of this list
#   risk_score_min       - risk_score must be >= this value
#   risk_score_max       - risk_score must be <= this value (loose band, since
#                          LLM indicator picks vary run to run)
# ---------------------------------------------------------------------------

TEST_CASES = [
    (
        "digital_arrest_case",
        "Hello Sir, I am from CBI. Your Aadhaar is linked with money "
        "laundering. Do not disconnect the call. Transfer Rs 2,00,000 "
        "immediately to avoid arrest.",
        {
            "fraud_type": "Digital Arrest Scam",
            "severity": "High",
            "min_indicators": [
                "Government Impersonation",
                "Criminal Allegation",
            ],
            "risk_score_min": 60,
            "risk_score_max": 100,
        },
    ),
    (
        "benign_case",
        "Hi, just checking if you are free for lunch tomorrow at 1pm.",
        {
            "fraud_type": "Unclassified Suspicious Activity",
            "severity": "Low",
            "min_indicators": [],
            "risk_score_min": 0,
            "risk_score_max": 0,
        },
    ),
    (
        "upi_payment_fraud_case",
        "You have received Rs 5,000 by mistake via UPI. Please transfer it "
        "back immediately to this UPI ID rahul.verify@upi or we will report "
        "you to the bank. Share the OTP sent to your phone to confirm the "
        "reversal right now.",
        {
            "fraud_type": "UPI / Payment Fraud",
            "severity": None,
            "min_indicators": ["OTP Request"],
            "risk_score_min": 20,
            "risk_score_max": 100,
        },
    ),
    (
        "phishing_credential_case",
        "Your bank account will be suspended. Verify immediately by "
        "clicking this link: http://secure-bank-verify.example.com and "
        "entering your OTP to keep your account active.",
        {
            "fraud_type": "Phishing / Credential Theft",
            "severity": None,
            "min_indicators": ["Suspicious Link", "OTP Request"],
            "risk_score_min": 20,
            "risk_score_max": 100,
        },
    ),
    (
        "extortion_threat_case",
        "We have obtained your private photos. Pay Rs 50,000 immediately "
        "or we will leak everything online and you will face a criminal "
        "case and non bailable arrest.",
        {
            "fraud_type": "Extortion / Threat-based Fraud",
            "severity": None,
            "min_indicators": ["Threat Language"],
            "risk_score_min": 30,
            "risk_score_max": 100,
        },
    ),
    (
        "ambiguous_weak_signal_case",
        "This is urgent, please call me back as soon as you can, it's "
        "important.",
        {
            "fraud_type": "Unclassified Suspicious Activity",
            "severity": None,
            "min_indicators": [],
            "risk_score_min": 0,
            "risk_score_max": 30,
        },
    ),
]

In [55]:
def _check_field(label: str, actual: Any, expected: Any) -> bool:
    if expected is None:
        return True  # not asserted for this case
    ok = actual == expected
    status = "PASS" if ok else "FAIL"
    print(f"    [{status}] {label}: expected={expected!r} actual={actual!r}")
    return ok

In [56]:
def _check_min_indicators(actual_indicators: List[str], expected_min: List[str]) -> bool:
    actual_set = set(actual_indicators)
    missing = [ind for ind in expected_min if ind not in actual_set]
    ok = not missing
    status = "PASS" if ok else "FAIL"
    print(
        f"    [{status}] min_indicators: expected subset={expected_min!r} "
        f"actual={actual_indicators!r}"
        + (f" MISSING={missing!r}" if missing else "")
    )
    return ok

In [57]:
def _check_range(label: str, actual: int, lo: int, hi: int) -> bool:
    ok = lo <= actual <= hi
    status = "PASS" if ok else "FAIL"
    print(f"    [{status}] {label}: expected in [{lo},{hi}] actual={actual}")
    return ok

In [58]:
def run_test_suite() -> Dict[str, Any]:
    """
    Runs every case in TEST_CASES through analyze_case() and compares the
    result against its expected_dict. Prints a PASS/FAIL breakdown per
    field per case, then a final summary line.
    """
    total_checks = 0
    passed_checks = 0
    case_results: Dict[str, Any] = {}

    for name, text, expected in TEST_CASES:
        print(f"\n=== {name} ===")
        print(f"Input: {text[:80]}{'...' if len(text) > 80 else ''}")

        try:
            result = analyze_case(text)
        except FraudIntelligenceError as exc:
            print(f"    [FAIL] analyze_case raised an error: {exc}")
            case_results[name] = {"error": str(exc)}
            total_checks += 1
            continue

        case_results[name] = result
        checks: List[bool] = []

        checks.append(
            _check_field("fraud_type", result["fraud_type"], expected["fraud_type"])
        )
        checks.append(
            _check_field("severity", result["severity"], expected["severity"])
        )
        checks.append(
            _check_min_indicators(result["indicators"], expected["min_indicators"])
        )
        checks.append(
            _check_range(
                "risk_score",
                result["risk_score"],
                expected["risk_score_min"],
                expected["risk_score_max"],
            )
        )

        total_checks += len(checks)
        passed_checks += sum(checks)

        print(f"    confidence={result['confidence']} "
              f"classification_match_ratio={result['classification_match_ratio']}")

    print("\n" + "=" * 60)
    print(f"SUMMARY: {passed_checks}/{total_checks} checks passed "
          f"across {len(TEST_CASES)} cases")
    print("=" * 60)

    return case_results

In [59]:
all_results = run_test_suite()
print("\nFull results (for inspection):")
print(json.dumps(all_results, indent=2, default=str))

2026-07-11 06:17:27,971 | INFO | fraud_intelligence_engine | Signal extraction complete. entities=1 behaviour_flags=5



=== digital_arrest_case ===
Input: Hello Sir, I am from CBI. Your Aadhaar is linked with money laundering. Do not d...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-11 06:17:28,473 | INFO | fraud_intelligence_engine | Retrieved 5 relevant chunks (of 5 candidates) for query.
2026-07-11 06:17:28,473 | INFO | fraud_intelligence_engine | Sending request to OpenRouter (tencent/hy3:free), attempt 1, max_tokens=4096.
2026-07-11 06:17:49,513 | INFO | fraud_intelligence_engine | LLM reasoning complete. indicators=4 evidence_sufficient=True
2026-07-11 06:17:49,513 | INFO | fraud_intelligence_engine | Fraud classification complete. chosen=Digital Arrest Scam best_ratio=0.80
2026-07-11 06:17:49,516 | INFO | fraud_intelligence_engine | Case e077c1f1-aab4-4d39-bee2-b3241e3c07c5 analysed. fraud_type=Digital Arrest Scam severity=High confidence=75
2026-07-11 06:17:49,517 | INFO | fraud_intelligence_engine | Signal extraction complete. entities=0 behaviour_flags=0


    [PASS] fraud_type: expected='Digital Arrest Scam' actual='Digital Arrest Scam'
    [PASS] severity: expected='High' actual='High'
    [PASS] min_indicators: expected subset=['Government Impersonation', 'Criminal Allegation'] actual=['Government Impersonation', 'Criminal Allegation', 'Urgent Payment Request', 'Isolation Tactics']
    [PASS] risk_score: expected in [60,100] actual=70
    confidence=75 classification_match_ratio=0.8

=== benign_case ===
Input: Hi, just checking if you are free for lunch tomorrow at 1pm.


...

2026-07-11 06:19:00,739 | INFO | fraud_intelligence_engine | Retrieved 5 relevant chunks (of 5 candidates) for query.
2026-07-11 06:19:00,739 | INFO | fraud_intelligence_engine | Sending request to OpenRouter (tencent/hy3:free), attempt 1, max_tokens=4096.
2026-07-11 06:19:40,100 | INFO | fraud_intelligence_engine | LLM reasoning complete. indicators=3 evidence_sufficient=True
2026-07-11 06:19:40,101 | INFO | fraud_intelligence_engine | Fraud classification complete. chosen=Digital Arrest Scam best_ratio=0.40
2026-07-11 06:19:40,102 | INFO | fraud_intelligence_engine | Case f747b3e9-7de2-48f3-83d2-1dea5863d5f7 analysed. fraud_type=Digital Arrest Scam severity=Medium confidence=65
2026-07-11 06:19:40,104 | INFO | fraud_intelligence_engine | Signal extraction complete. entities=0 behaviour_flags=1


    [FAIL] fraud_type: expected='Extortion / Threat-based Fraud' actual='Digital Arrest Scam'
    [PASS] min_indicators: expected subset=['Threat Language'] actual=['Government Impersonation', 'Threat Language', 'Urgency']
    [PASS] risk_score: expected in [30,100] actual=60
    confidence=65 classification_match_ratio=0.4

=== ambiguous_weak_signal_case ===
Input: This is urgent, please call me back as soon as you can, it's important.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-11 06:19:40,514 | INFO | fraud_intelligence_engine | Retrieved 5 relevant chunks (of 5 candidates) for query.
2026-07-11 06:19:40,514 | INFO | fraud_intelligence_engine | Sending request to OpenRouter (tencent/hy3:free), attempt 1, max_tokens=4096.
2026-07-11 06:19:56,288 | INFO | fraud_intelligence_engine | LLM reasoning complete. indicators=1 evidence_sufficient=False
2026-07-11 06:19:56,288 | INFO | fraud_intelligence_engine | Fraud classification complete. chosen=Unclassified Suspicious Activity best_ratio=0.33
2026-07-11 06:19:56,288 | INFO | fraud_intelligence_engine | Case ca07d6d2-c36d-4047-b002-a335748a17ed analysed. fraud_type=Unclassified Suspicious Activity severity=Low confidence=15


    [PASS] fraud_type: expected='Unclassified Suspicious Activity' actual='Unclassified Suspicious Activity'
    [PASS] min_indicators: expected subset=[] actual=['Urgency']
    [PASS] risk_score: expected in [0,30] actual=15
    confidence=15 classification_match_ratio=0.3333

SUMMARY: 23/24 checks passed across 6 cases

Full results (for inspection):
{
  "digital_arrest_case": {
    "case_id": "e077c1f1-aab4-4d39-bee2-b3241e3c07c5",
    "timestamp": "2026-07-11T00:47:27.971704+00:00",
    "fraud_type": "Digital Arrest Scam",
    "classification_match_ratio": 0.8,
    "confidence": 75,
    "severity": "High",
    "risk_score": 70,
    "risk_breakdown": {
      "Government Impersonation": 25,
      "Criminal Allegation": 20,
      "Urgent Payment Request": 15,
      "Isolation Tactics": 10
    },
    "indicators": [
      "Government Impersonation",
      "Criminal Allegation",
      "Urgent Payment Request",
      "Isolation Tactics"
    ],
    "entities": {
      "money_amounts": [
 